# Silver CSV Load Notebook

この Notebook は、別 Lakehouse にアップロードした CSV を読み込み、`silver` スキーマのテーブルとして保存します。

## 想定入力
- `Files/3.2-sales-model/silver-export/City.csv`
- `Files/3.2-sales-model/silver-export/Sale.csv`
- `Files/3.2-sales-model/silver-export/Stock_Holding.csv`
- `Files/3.2-sales-model/silver-export/Stock_Item.csv`

## 作成されるテーブル
- `silver.City`
- `silver.Sale`
- `silver.Stock_Holding`
- `silver.Stock_Item`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, TimestampType, DateType

# 必要に応じて変更してください
TARGET_SCHEMA = "silver"
CSV_BASE_PATH = "Files/3.2-sales-model/silver-export"
TABLE_NAMES = ["City", "Sale", "Stock_Holding", "Stock_Item"]

TABLE_CONFIGS = {
    table_name: {
        "csv": f"{CSV_BASE_PATH}/{table_name}.csv",
        "path": f"Tables/{TARGET_SCHEMA}/{table_name}"
    }
    for table_name in TABLE_NAMES
}

print("TARGET_SCHEMA:", TARGET_SCHEMA)
print("CSV_BASE_PATH:", CSV_BASE_PATH)
TABLE_CONFIGS

## スキーマ作成
ロード先の `silver` スキーマを作成または確認します。

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")
display(spark.sql(f"DESCRIBE SCHEMA {TARGET_SCHEMA}"))

## テーブル別変換関数
CSV の推定型を補正し、元の Silver テーブルに近い型へ整えます。

In [ ]:
def read_csv(csv_path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(csv_path)
    )

def transform_city(df):
    return (
        df
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("WWI_City_ID", F.col("WWI_City_ID").cast(IntegerType()))
        .withColumn("Latest_Recorded_Population", F.col("Latest_Recorded_Population").cast(IntegerType()))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
    )

def transform_sale(df):
    return (
        df
        .withColumn("Sale_Key", F.col("Sale_Key").cast(IntegerType()))
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("Customer_Key", F.col("Customer_Key").cast(IntegerType()))
        .withColumn("Bill_To_Customer_Key", F.col("Bill_To_Customer_Key").cast(IntegerType()))
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("Salesperson_Key", F.col("Salesperson_Key").cast(IntegerType()))
        .withColumn("WWI_Invoice_ID", F.col("WWI_Invoice_ID").cast(IntegerType()))
        .withColumn("Invoice_Date_Key", F.to_date("Invoice_Date_Key"))
        .withColumn("Delivery_Date_Key", F.to_date("Delivery_Date_Key"))
        .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(DoubleType()))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(DoubleType()))
        .withColumn("Total_Excluding_Tax", F.col("Total_Excluding_Tax").cast(DoubleType()))
        .withColumn("Tax_Amount", F.col("Tax_Amount").cast(DoubleType()))
        .withColumn("Profit", F.col("Profit").cast(DoubleType()))
        .withColumn("Total_Including_Tax", F.col("Total_Including_Tax").cast(DoubleType()))
        .withColumn("Total_Dry_Items", F.col("Total_Dry_Items").cast(DoubleType()))
        .withColumn("Total_Chiller_Items", F.col("Total_Chiller_Items").cast(DoubleType()))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("ProcessedTimestamp", F.to_timestamp("ProcessedTimestamp"))
    )

def transform_stock_holding(df):
    return (
        df
        .withColumn("Stock_Holding_Key", F.col("Stock_Holding_Key").cast(IntegerType()))
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("Quantity_On_Hand", F.col("Quantity_On_Hand").cast(IntegerType()))
        .withColumn("Last_Stocktake_Quantity", F.col("Last_Stocktake_Quantity").cast(IntegerType()))
        .withColumn("Last_Cost_Price", F.col("Last_Cost_Price").cast(DoubleType()))
        .withColumn("Reorder_Level", F.col("Reorder_Level").cast(IntegerType()))
        .withColumn("Target_Stock_Level", F.col("Target_Stock_Level").cast(IntegerType()))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
    )

def transform_stock_item(df):
    return (
        df
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("WWI_Stock_Item_ID", F.col("WWI_Stock_Item_ID").cast(IntegerType()))
        .withColumn("Lead_Time_Days", F.col("Lead_Time_Days").cast(IntegerType()))
        .withColumn("Quantity_Per_Outer", F.col("Quantity_Per_Outer").cast(IntegerType()))
        .withColumn("Is_Chiller_Stock", F.col("Is_Chiller_Stock").cast(BooleanType()))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(DoubleType()))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(DoubleType()))
        .withColumn("Recommended_Retail_Price", F.col("Recommended_Retail_Price").cast(DoubleType()))
        .withColumn("Typical_Weight_Per_Unit", F.col("Typical_Weight_Per_Unit").cast(DoubleType()))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
    )

TRANSFORMERS = {
    "City": transform_city,
    "Sale": transform_sale,
    "Stock_Holding": transform_stock_holding,
    "Stock_Item": transform_stock_item,
}

## CSV のロードとテーブル保存
各 CSV を読み込み、`Tables/silver/*` に保存します。

In [ ]:
load_results = []
for table_name, config in TABLE_CONFIGS.items():
    print(f"Loading {config['csv']}")
    df = read_csv(config['csv'])
    transformed = TRANSFORMERS[table_name](df)
    transformed.write.mode("overwrite").save(config['path'])
    load_results.append((table_name, config['path'], transformed.count()))

display(spark.createDataFrame(load_results, ["TableName", "Path", "RowCount"]))

## 読み込み結果の確認
ロード後のテーブルを確認します。

In [ ]:
for table_name in TABLE_NAMES:
    print(f"Preview: {TARGET_SCHEMA}.{table_name}")
    display(spark.read.format("delta").load(TABLE_CONFIGS[table_name]['path']).limit(10))

## 補足
この Notebook は CSV をそのまま別 Lakehouse に復元する用途を想定しています。Chapter 3 用に Gold モデルを作る場合は、このあと別 Notebook で `silver` から `gold` に変換してください。